# Text-to-speech with ESPnet-TTS

Synthesise speech with pretrained single-speaker and multi-speaker models,
compare text2wav against text2mel plus a vocoder, condition on a speaker
embedding, and score the result with VERSA.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet-TTS recipe template](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/tts1)
- [VERSA](https://github.com/wavlab-speech/versa)


## Installation

Apart from ESPNet, we need to install additional modules for vocoder, text preprocessing, and audio evalaution.


In [ ]:
# Around 5-6 minutes

# NOTE: pip shows imcompatible errors due to preinstalled libraries but you do not need to care
%pip install -q "espnet[tts,spk] @ git+https://github.com/espnet/espnet"

# Vocoder
!git clone --depth 5 https://github.com/ftshijt/ParallelWaveGAN.git
!cd ParallelWaveGAN && pip install .

# text preprocessing
!pip install pypinyin==0.44.0
!pip install gdown==4.4.0
!pip install ipywebrtc

# Evaluation related
%pip install -q levenshtein
!git clone --depth 5 https://github.com/wavlab-speech/versa.git
!cd versa && pip install .
!git clone https://github.com/ftshijt/versa_demo_egs.git

# text preprocessing
import nltk
nltk.download('averaged_perceptron_tagger_eng')

# Create a folder for the synthesized results and further evaluation
!mkdir tts_result

tacotron_text = "None"
fastspeech_text = "None"
vits_text = "None"

# parallel_wavegan 0.6.1, the newest release, imports scipy.signal.kaiser,
# which SciPy removed in 1.13 - it lives in scipy.signal.windows now. Put it
# back before anything imports the vocoder, or the text2mel path fails on any
# current SciPy.
import scipy.signal

if not hasattr(scipy.signal, "kaiser"):
    from scipy.signal.windows import kaiser as _kaiser

    scipy.signal.kaiser = _kaiser

import torch

# VERSA runs as a separate process and takes --device; it dropped the older
# --use_gpu flag, which is what this notebook used to pass
VERSA_DEVICE = "gpu" if torch.cuda.is_available() else "cpu"


## Single speaker TTS model demo


### TTS Model

You can try end-to-end text2wav model & combination of text2mel and vocoder.
If you use text2wav model, you do not need to use vocoder (automatically disabled).

**Text2wav models**:
- VITS

**Text2mel models**:
- Tacotron2
- Transformer-TTS
- (Conformer) FastSpeech
- (Conformer) FastSpeech2

**Vocoders**:
- Parallel WaveGAN
- Multi-band MelGAN
- HiFiGAN
- Style MelGAN.

In this demo, we will only experiment with the English TTS model, but ESPnet-TTS supports multiple languages like Japanese and Mandarin.

> The terms of use follow that of each corpus. ESPnet-TTS use the following corpora:
- `ljspeech_*`: LJSpeech dataset
  - https://keithito.com/LJ-Speech-Dataset/
- `jsut_*`: JSUT corpus
  - https://sites.google.com/site/shinnosuketakamichi/publication/jsut
- `jvs_*`: JVS corpus + JSUT corpus
  - https://sites.google.com/site/shinnosuketakamichi/research-topics/jvs_corpus
  - https://sites.google.com/site/shinnosuketakamichi/publication/jsut
- `tsukuyomi_*`: つくよみちゃんコーパス + JSUT corpus
  - https://tyc.rei-yumesaki.net/material/corpus/
  - https://sites.google.com/site/shinnosuketakamichi/publication/jsut
- `csmsc_*`: Chinese Standard Mandarin Speech Corpus
  - https://www.data-baker.com/open_source.html


In [ ]:
#@title Download English model { run: "auto" }
lang = 'English'
tag = "kan-bayashi/ljspeech_tacotron2" #@param ["kan-bayashi/ljspeech_tacotron2"]
vocoder_tag = "parallel_wavegan/ljspeech_parallel_wavegan.v1" #@param ["parallel_wavegan/ljspeech_parallel_wavegan.v1"]

### Model Setup


🔍 **Possible Exploration:** Try tuning the parameters in `Text2Speech.from_pretrained()` and observe the difference in generated speech


In [ ]:
import torch
from espnet2.bin.tts_inference import Text2Speech
from espnet2.utils.types import str_or_none

text2speech = Text2Speech.from_pretrained(
    model_tag=tag,
    vocoder_tag=str_or_none(vocoder_tag),
    device="cuda" if torch.cuda.is_available() else "cpu",
    # Only for Tacotron 2 & Transformer
    threshold=0.5,
    # Only for Tacotron 2
    minlenratio=0.0,
    maxlenratio=10.0,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    # Only for FastSpeech & FastSpeech2 & VITS (not used in this block)
    speed_control_alpha=1.0,
    # Only for VITS (not used in this block)
    noise_scale=0.333,
    noise_scale_dur=0.333,
)

### Synthesis  )

Run inference of pretrained single-speaker TTS model. Please experiment with running TTS model on different utterances. Provide at least one example of failure cases, plot their spectrogram and waveform, and discuss their possible causes.


🔍 **Possible Exploration:** Try different input text and observe the generated speech:
- normalized text vs raw text (like numbers and abbreviations)
- same sentence with different punctuation (like commas, semicolons, ellipses, line breaks)
- out-of-domain text styles (like a short casual sentence or a long formal paragraph)
- other languages or code-switching


In [ ]:
import time
import torch

# decide the input sentence by yourself
x = "This is a demonstration of text to speech with ESPnet."
# ^ edit this and run the cell again
tacotron_text = x

# synthesis
with torch.no_grad():
    start = time.time()
    wav = text2speech(x)["wav"]
rtf = (time.time() - start) / (len(wav) / text2speech.fs)
print(f"RTF = {rtf:5f}")

# let us listen to generated samples
from IPython.display import display, Audio
display(Audio(wav.view(-1).cpu().numpy(), rate=text2speech.fs))

# let us save the synthesized speech for further analysis later
import soundfile as sf
sf.write("tts_result/tacotron_parallelwavegan.wav", wav.view(-1).cpu().numpy(), text2speech.fs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from espnet2.layers.stft import Stft

sr = text2speech.fs
stft = Stft(n_fft=512, win_length=None, hop_length=128, window="hann")
wav_cpu = wav.view(-1).cpu()
spec = stft(wav_cpu.unsqueeze(0), torch.LongTensor([len(wav_cpu)]))[0].squeeze()
magnitude = torch.linalg.norm(spec, dim=-1).transpose(-1, -2).numpy()

fig, (left, right) = plt.subplots(1, 2, figsize=(18, 4))
left.imshow(
    20 * np.log10(np.maximum(magnitude, 1e-8)),
    origin="lower",
    aspect="auto",
    extent=[0, len(wav_cpu) / sr, 0, sr / 2],
    cmap="magma",
)
left.set_title("Spectrogram")
left.set_ylabel("Frequency (Hz)")
right.plot(torch.linspace(0, len(wav_cpu) / sr, len(wav_cpu)), wav_cpu)
right.set_xlim(0, len(wav_cpu) / sr)
right.set_title("Waveform")
right.set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


### TTS Model selection


### Question2  )

Please experiment with running different TTS models like FastSpeech2. Discuss which is better and why.


In [ ]:
#@title Download English model { run: "auto" }
lang = 'English'
tag = "kan-bayashi/ljspeech_conformer_fastspeech2" # @param ["kan-bayashi/ljspeech_conformer_fastspeech2", "kan-bayashi/ljspeech_fastspeech"]
vocoder_tag = "parallel_wavegan/ljspeech_parallel_wavegan.v1" #@param ["parallel_wavegan/ljspeech_parallel_wavegan.v1", "none"]
# when vocoder_tag is none, Griffin Lim algorithm is used

In [ ]:
import torch
# For fastspeech model run the commented lines below
from espnet2.bin.tts_inference import Text2Speech
from espnet2.utils.types import str_or_none
!ln -sf tts_fastspeech_model/exp .
text2speech = Text2Speech.from_pretrained(
    model_tag=tag,
    vocoder_tag=str_or_none(vocoder_tag),
    device="cuda" if torch.cuda.is_available() else "cpu",
    # Only for FastSpeech & FastSpeech2 & VITS, increasing alpha will make the generated speech slower
    speed_control_alpha=1.0,
)

In [ ]:
import time
import torch

# decide the input sentence by yourself
x = "This is a demonstration of text to speech with ESPnet."
# ^ edit this and run the cell again
fastspeech_text = x

# synthesis
with torch.no_grad():
    start = time.time()
    wav = text2speech(x)["wav"]
rtf = (time.time() - start) / (len(wav) / text2speech.fs)
print(f"RTF = {rtf:5f}")

# let us listen to generated samples
from IPython.display import display, Audio
display(Audio(wav.view(-1).cpu().numpy(), rate=text2speech.fs))

# let us save the synthesized speech for further analysis later
import soundfile as sf
sf.write("tts_result/fastspeech_parallelwavegan.wav", wav.view(-1).cpu().numpy(), text2speech.fs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from espnet2.layers.stft import Stft

sr = text2speech.fs
stft = Stft(n_fft=512, win_length=None, hop_length=128, window="hann")
wav_cpu = wav.view(-1).cpu()
spec = stft(wav_cpu.unsqueeze(0), torch.LongTensor([len(wav_cpu)]))[0].squeeze()
magnitude = torch.linalg.norm(spec, dim=-1).transpose(-1, -2).numpy()

fig, (left, right) = plt.subplots(1, 2, figsize=(18, 4))
left.imshow(
    20 * np.log10(np.maximum(magnitude, 1e-8)),
    origin="lower",
    aspect="auto",
    extent=[0, len(wav_cpu) / sr, 0, sr / 2],
    cmap="magma",
)
left.set_title("Spectrogram")
left.set_ylabel("Frequency (Hz)")
right.plot(torch.linspace(0, len(wav_cpu) / sr, len(wav_cpu)), wav_cpu)
right.set_xlim(0, len(wav_cpu) / sr)
right.set_title("Waveform")
right.set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


## Multi-speaker Model Demo


### Model Selection

Now we provide only English multi-speaker pretrained model.

> The terms of use follow that of each corpus. We use the following corpus:
- `vctk_*`: English Multi-speaker Corpus for CSTR Voice Cloning Toolkit
  - http://www.udialogue.org/download/cstr-vctk-corpus.html


🔍 **Possible Exploration:** How do naturalness and speaker similarity of the generated speech change under:
- different text input (like short or long text, rare words)
- different reference speaker voice (like different duration, different spoken content)


### Model Setup


In [ ]:
import torch
from espnet2.bin.tts_inference import Text2Speech
from espnet2.utils.types import str_or_none

text2speech = Text2Speech.from_pretrained(
    model_tag="espnet/espnet_tts_vctk_espnet_spk_voxceleb12_rawnet",
    device="cuda" if torch.cuda.is_available() else "cpu",
    # Only for Tacotron 2 & Transformer
    threshold=0.5,
    # Only for Tacotron 2
    minlenratio=0.0,
    maxlenratio=10.0,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    # Only for FastSpeech & FastSpeech2 & VITS
    speed_control_alpha=1.0,
    # Only for VITS
    noise_scale=0.333,
    noise_scale_dur=0.333,
)

### Speaker selection

For multi-speaker model, we need to provide speaker embedding to the system to achieve zero-shot multi-speaker styles. To get the speaker embedding, you can either record your own voice or use existing sample audio. Please pick up your options by optionally execute the code blocks below.


In [ ]:
# or if you would like to use our provided voice,
# please select one from the three utterances from the examples

import librosa

# choose one reference speaker voice
reference_audio_path = "versa_demo_egs/examples/spk/spk1/1.wav" # @param ["versa_demo_egs/examples/spk/spk1/1.wav", "versa_demo_egs/examples/spk/spk2/1.wav"]
audio, sr = librosa.load(reference_audio_path)

# use sampling rate as 16000
audio = librosa.resample(audio,orig_sr=sr, target_sr=16000)

After we get the reference audio, we can extract speaker embedding with a pre-trained model.


In [ ]:
from espnet2.bin.spk_inference import Speech2Embedding

# load espnet speaker embedding
speech2spk_embed = Speech2Embedding.from_pretrained(model_tag="espnet/voxcelebs12_rawnet3")

spembs = speech2spk_embed(audio)

### Synthesis)

Run inference of pretrained multi-speaker TTS model on more than one reference speakers. Plot spectrogram and waveform of the synthesized speech for these speakers.


In [ ]:
import time
import torch

# decide the input sentence by yourself
x = "This is a demonstration of text to speech with ESPnet."
# ^ edit this and run the cell again
vits_text = x

# synthesis
with torch.no_grad():
    start = time.time()
    print(spembs.shape)
    wav = text2speech(x, spembs=spembs.squeeze(0))["wav"]
rtf = (time.time() - start) / (len(wav) / text2speech.fs)
print(f"RTF = {rtf:5f}")

# let us listen to generated samples
from IPython.display import display, Audio
display(Audio(wav.view(-1).cpu().numpy(), rate=text2speech.fs))

# let us save the synthesized speech for further analysis later
import soundfile as sf
sf.write("tts_result/vits.wav", wav.view(-1).cpu().numpy(), text2speech.fs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from espnet2.layers.stft import Stft

sr = text2speech.fs
stft = Stft(n_fft=512, win_length=None, hop_length=128, window="hann")
wav_cpu = wav.view(-1).cpu()
spec = stft(wav_cpu.unsqueeze(0), torch.LongTensor([len(wav_cpu)]))[0].squeeze()
magnitude = torch.linalg.norm(spec, dim=-1).transpose(-1, -2).numpy()

fig, (left, right) = plt.subplots(1, 2, figsize=(18, 4))
left.imshow(
    20 * np.log10(np.maximum(magnitude, 1e-8)),
    origin="lower",
    aspect="auto",
    extent=[0, len(wav_cpu) / sr, 0, sr / 2],
    cmap="magma",
)
left.set_title("Spectrogram")
left.set_ylabel("Frequency (Hz)")
right.plot(torch.linspace(0, len(wav_cpu) / sr, len(wav_cpu)), wav_cpu)
right.set_xlim(0, len(wav_cpu) / sr)
right.set_title("Waveform")
right.set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


## Evaluation of TTS Systems

In this demonstration, we explore the use of VERSA, a modern evaluation tool for text-to-speech (TTS) systems. Unlike traditional methods that focus on limited aspects like intelligibility or error rates, VERSA uses deep learning to capture more nuanced qualities of speech such as naturalness, expressiveness, and overall audio quality. This approach allows us to more accurately compare different TTS models and better understand their strengths and weaknesses.

Reference:
- [VERSA demonstration](https://colab.research.google.com/drive/11c0vZxbSa8invMSfqM999tI3MnyAVsOp?usp=sharing)
- [VERSA repository](https://github.com/shinjiwlab/versa)
- [VERSA paper](https://arxiv.org/abs/2412.17667)

Let's start with an example of VERSA


In [ ]:
! gdown 1ReqBVGH-_eM0kyqLorit4kWyeIXtwm_H
!cat tts_demo_11492_16k.yaml

In our next step, let's start to evalaute the TTS results with VERSA.


In [ ]:
# Generate a transcription file for ASR-based evaluation

with open("transcription.txt", "w") as f:
  f.write("tacotron_parallelwavegan.wav {}\n".format(tacotron_text))
  f.write("fastspeech_parallelwavegan.wav {}\n".format(fastspeech_text))
  f.write("vits.wav {}\n".format(vits_text))

We then use VERSA to score the given data.


In [ ]:
# Score the TTS results with VERSA

# As many pre-trained models are downloaded, the initial start may takes around
# 2-3 minutes
! python -m versa.bin.scorer \
    --score_config tts_demo_11492_16k.yaml \
    --pred tts_result \
    --io dir \
    --text transcription.txt \
    --device {VERSA_DEVICE} \
    --output_file result.json

In [ ]:
# This is the raw format of the results
print("*" * 10, "Raw Format", "*" * 10)
!cat result.json


print("*" * 10, "Better Format", "*" * 10)
# We can make it easier to visualize as follows:
import ast, json
data = []
with open("result.json", 'r') as infile:
    for line in infile:
        line = line.strip()
        if not line:
            continue  # Skip empty lines
        try:
            # Safely evaluate the line to convert it into a Python dict.
            record = ast.literal_eval(line)
            # Round float values to two decimals.
            for key, value in record.items():
                if isinstance(value, float):
                    record[key] = round(value, 2)
            # calculate utterance-cer/wer
            if "owsm_hyp_text" in record.keys():
                record["owsm_cer"] = (
                      record["owsm_cer_delete"] +
                      record["owsm_cer_replace"] +
                      record["owsm_cer_insert"]
                  ) / (
                      record["owsm_cer_replace"] +
                      record["owsm_cer_equal"] +
                      record["owsm_cer_delete"]
                  )
                record["owsm_wer"] = (
                      record["owsm_wer_delete"] +
                      record["owsm_wer_replace"] +
                      record["owsm_wer_insert"]
                  ) / (
                      record["owsm_wer_replace"] +
                      record["owsm_wer_equal"] +
                      record["owsm_wer_delete"]
                  )
            data.append(record)
        except Exception as e:
            print(f"Error parsing line:\n{line}\n{e}")

# Print the beautified JSON directly.
print(json.dumps(data, indent=4))